# Dates and Times in Python and Pandas

This is a companion notebook to a [Substack article](https://thedatafieldbook.substack.com/p/dates-and-times-in-python-and-pandas) with the same title. The purpose of this notebook is to create a **dummy dataset** and walk through the essentials of working with **date/time values** in **Python and Pandas**.

**Topics covered:** parsing, time zones, `dt` accessor, filtering, resampling, rolling windows, offsets, periods, intervals, `merge_asof`, rounding, and common pitfalls.

In [ ]:
import pandas as pd
import numpy as np
import platform
from datetime import date, time, datetime, timedelta, timezone

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 50)

In [ ]:
print("python version:", platform.python_version())

In [ ]:
print("pandas version:", pd.__version__)

In [ ]:
import sys, pandas as pd
print("Python:", sys.version)
print("Python executable:", sys.executable)
print("Pandas:", pd.__version__)

In [ ]:
help(datetime)

Exploring DateTime objects in Python

In [ ]:
# Today's date
today = date.today()
print("Today:", today)

In [ ]:
# Specific date
d = date(2025, 2, 8)
print("Specific date:", d)

In [ ]:
# Time only
t = time(14, 30, 15)
print("Time:", t)

In [ ]:
import datetime
# Current date & time
now_dt = datetime.datetime.now()
print("Now:", now_dt)

In [ ]:
print(type(now_dt))

In [ ]:
now_dt = now_dt.replace(tzinfo=timezone.utc)

In [ ]:
print(type(now_dt))

In [ ]:
timestamp = dt.timestamp()
print(timestamp)

In [ ]:
print(type(timestamp))

In [ ]:
print(now.year, now.month, now.day, now.hour, now.minute)

In [ ]:
today = date.today()

tomorrow = today + timedelta(days=1)
last_week = today - timedelta(weeks=1)

print("Tomorrow:", tomorrow)
print("Last week:", last_week)

# Add hours to datetime
now = datetime.now()
in_two_hours = now + timedelta(hours=2)

print("Now:", now)
print("In 2 hours:", in_two_hours)

In [ ]:
from datetime import date

d1 = date(2025, 1, 1)
d2 = date(2025, 2, 8)

delta = d2 - d1
print("Days between:", delta.days)


In [ ]:
now = datetime.now()

print(now.strftime("%Y-%m-%d"))
print(now.strftime("%d/%m/%Y"))
print(now.strftime("%A, %B %d, %Y"))
print(now.strftime("%H:%M:%S"))


In [ ]:
from datetime import datetime

text = "2025-02-08 14:30:15"
dt = datetime.strptime(text, "%Y-%m-%d %H:%M:%S")

print(dt)
print(type(dt))


In [ ]:
print(dt.year, dt.month, dt.day)


In [ ]:
now = datetime.now()

iso = now.isoformat()
print("ISO:", iso)

parsed = datetime.fromisoformat(iso)
print("Parsed:", parsed)


In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo

utc_now = datetime.now(tz=ZoneInfo("UTC"))
dublin_now = datetime.now(tz=ZoneInfo("Europe/Dublin"))
new_york_now = datetime.now(tz=ZoneInfo("America/New_York"))

print("UTC:", utc_now)
print("Dublin:", dublin_now)
print("New York:", new_york_now)


In [ ]:
dublin_as_ny = dublin_now.astimezone(ZoneInfo("America/New_York"))
print("Dublin converted to NY:", dublin_as_ny)

In [ ]:
naive = datetime.now()
aware = datetime.now(tz=ZoneInfo("UTC"))

print("Naive:", naive, naive.tzinfo)
print("Aware:", aware, aware.tzinfo)


In [ ]:
from datetime import date

birth = date(1985, 1, 14)
today = date.today()
age = today.year - birth.year - ((today.month, today.day) < (birth.month, birth.day))
print("Age:", age)


In [ ]:
from datetime import datetime
import time

start = datetime.now()
time.sleep(1.5)
end = datetime.now()

print("Elapsed:", end - start)


## 1) Create a sample DataFrame with messy date/time values
We'll intentionally include:
- ISO strings
- custom formats
- mixed time zones
- invalid / missing values
- separate date + time columns


In [ ]:
pd.set_option("display.width", 120)

rng = np.random.default_rng(42)

n = 40
base = pd.Timestamp("2025-01-01")

# A clean timestamp index (naive)
ts_clean = base + pd.to_timedelta(rng.integers(0, 60*24*30, size=n), unit="m")
ts_clean = ts_clean.sort_values()

# Messy string representations
ts_str_iso = ts_clean.astype(str)
ts_str_custom = pd.Index(ts_clean).strftime("%d/%m/%Y %H:%M")  # day-first style

# Inject some mess: missing, invalid, different format
ts_str_mixed = np.array(ts_str_iso, dtype=object)
for i in rng.choice(n, size=6, replace=False):
    ts_str_mixed[i] = ts_str_custom[i]
ts_str_mixed[rng.integers(0, n)] = None
ts_str_mixed[rng.integers(0, n)] = "not a date"

# A separate date + time split
date_part = ts_clean.date.astype(str)
time_part = pd.Index(ts_clean).strftime("%H:%M:%S")

# Some categories and values
regions = rng.choice(["EU", "US", "APAC"], size=n, p=[0.5, 0.3, 0.2])
sales = rng.normal(loc=200, scale=50, size=n).round(2)

df = pd.DataFrame({
    "order_id": [f"ORD-{1000+i}" for i in range(n)],
    "region": regions,
    "sales": sales,
    "ordered_at_raw": ts_str_mixed,  # messy
    "date_str": date_part,
    "time_str": time_part,
})

df.head(10)


## 2) Convert strings to datetime safely with `pd.to_datetime`
Key parameters:
- `errors='coerce'` converts invalid parses to `NaT`
- `dayfirst=True` helps with `DD/MM/YYYY` formats
- `format=...` can improve speed/robustness when formats are consistent


In [ ]:
# Mixed formats: let pandas infer; handle invalid with coerce.
df["ordered_at"] = pd.to_datetime(df["ordered_at_raw"], errors="coerce", dayfirst=True)

df[["ordered_at_raw", "ordered_at"]].head(12)


In [ ]:
# How many failed parses?
df["ordered_at"].isna().sum(), len(df)


### Optional: strict parsing with an explicit format
This is faster and safer when your format is consistent.

In [ ]:
example = pd.Series(["06/02/2025 14:30", "07/02/2025 09:10"])
pd.to_datetime(example, format="%d/%m/%Y %H:%M")


## 3) Combine separate date and time columns
Common pattern: `date_str` + `time_str` -> a single timestamp column.

In [ ]:
df["ordered_at_from_parts"] = pd.to_datetime(df["date_str"] + " " + df["time_str"], errors="coerce")
df[["date_str", "time_str", "ordered_at_from_parts"]].head()


## 4) Datetime dtypes, `NaT`, and comparisons
- `datetime64[ns]` is pandas' primary datetime dtype
- Missing datetimes use `NaT` (Not-a-Time)
- Comparisons with `NaT` behave like `NaN` (propagates missingness)

In [ ]:
df["ordered_at"].dtype, df["ordered_at"].head()


In [ ]:
# Example comparisons & missing values
mask_after = df["ordered_at"] >= "2025-01-10"
mask_after.head(12), mask_after.isna().sum()


## 5) The `.dt` accessor (extract components)
Use `.dt` to pull calendar/time fields from a datetime series.

In [ ]:
dt = df["ordered_at"]

df["year"] = dt.dt.year
df["month"] = dt.dt.month
df["day"] = dt.dt.day
df["weekday_name"] = dt.dt.day_name()
df["hour"] = dt.dt.hour
df["date_only"] = dt.dt.date

df[["ordered_at", "year", "month", "day", "weekday_name", "hour", "date_only"]].head(10)


### Useful flags: weekend / business day-ish

In [ ]:
df["is_weekend"] = df["ordered_at"].dt.weekday >= 5
df[["ordered_at", "weekday_name", "is_weekend"]].head(10)


## 6) Sorting, indexing, and slicing by date
Datetime indexing unlocks convenient slicing and resampling.

In [ ]:
df_sorted = df.sort_values("ordered_at").copy()
df_sorted = df_sorted.set_index("ordered_at")

df_sorted.head(8)


In [ ]:
cols = ["order_id", "region", "sales"]

start = pd.Timestamp("2025-01-05")
end = pd.Timestamp("2025-01-12") + pd.Timedelta(days=1)

mask = (
    df_sorted.index.notna() &
    (df_sorted.index >= start) &
    (df_sorted.index < end)
)

out = df_sorted.loc[mask, cols].head(10)
out


## 7) Filtering with date logic
Examples: last 7 days, month boundaries, time-of-day filters.

In [ ]:
# Compute relative ranges using offsets
max_ts = df_sorted.index.max()
min_ts = df_sorted.index.min()

max_ts, min_ts


In [ ]:
# Ensure max_ts is a Timestamp (and not NaT)
max_ts = df_sorted.index.max()

start = max_ts - pd.Timedelta(days=7)
end = max_ts

mask = df_sorted.index.notna() & (df_sorted.index >= start) & (df_sorted.index <= end)

last_7_days = df_sorted.loc[mask, ["order_id", "region", "sales"]]
last_7_days.head(10)


In [ ]:
# Time-of-day filter: orders placed between 09:00 and 17:00
business_hours = df_sorted.between_time("09:00", "17:00")
business_hours[["order_id", "region", "sales"]].head()


## 8) Time zones: localize vs convert
Two common operations:
- `tz_localize` attaches a timezone to **naive** timestamps
- `tz_convert` converts between time zones for **aware** timestamps

We'll treat `ordered_at_from_parts` as a local time in Dublin, then convert to UTC and US/Eastern.

In [ ]:
# Start from naive times
s = pd.Series(df["ordered_at_from_parts"]).dropna().sort_values().reset_index(drop=True)

dublin = s.dt.tz_localize("Europe/Dublin")     # attach tz
utc = dublin.dt.tz_convert("UTC")              # convert tz
eastern = dublin.dt.tz_convert("US/Eastern")   # convert tz

out = pd.DataFrame({"Dublin": dublin, "UTC": utc, "US/Eastern": eastern})
out.head(5)


### Ambiguous and nonexistent times (DST)
During daylight saving transitions, some local times are ambiguous/nonexistent.
You can control behavior via `ambiguous=` and `nonexistent=`.

Example below uses US/Eastern around DST start (a nonexistent hour).

In [ ]:
dst_times = pd.to_datetime(["2025-03-09 01:30", "2025-03-09 02:30", "2025-03-09 03:30"])
# Localize with rules for nonexistent times
dst_localized = dst_times.tz_localize("US/Eastern", nonexistent="shift_forward")
dst_localized


## 9) Resampling time series data
Resampling aggregates values into fixed frequency buckets (hourly, daily, weekly, etc.).

In [ ]:
# Use the datetime index we created earlier
daily_sales = df_sorted["sales"].resample("D").sum()
daily_sales.head(10)


In [ ]:
weekly_sales = df_sorted["sales"].resample("W-MON").sum()  # weeks starting Monday
weekly_sales


### Resample with multiple columns and groupby
Example: daily sales per region.

In [ ]:
daily_region = df_sorted.groupby("region")["sales"].resample("D").sum().reset_index()
daily_region.head(12)


## 10) Rolling windows over time
Rolling computations can be based on a fixed window size (e.g., 7 rows) or a time window (e.g., '7D').

In [ ]:
# Time-based rolling: needs a datetime index
rolling_7d = daily_sales.rolling("7D").mean()
pd.DataFrame({"daily_sum": daily_sales, "rolling_7d_mean": rolling_7d}).head(15)


## 11) Differences, elapsed time, and durations
- `diff()` for elapsed time between events
- `Timedelta` arithmetic
- Use `.dt.total_seconds()` when you need numeric durations

In [ ]:
ordered = df_sorted.reset_index().sort_values("ordered_at")
ordered["prev_ordered_at"] = ordered["ordered_at"].shift(1)
ordered["gap"] = ordered["ordered_at"] - ordered["prev_ordered_at"]
ordered[["ordered_at", "prev_ordered_at", "gap"]].head(10)


In [ ]:
# Convert timedeltas to minutes
ordered["gap_minutes"] = ordered["gap"].dt.total_seconds() / 60
ordered[["gap", "gap_minutes"]].describe()


## 12) Offsets and date ranges
Offsets represent calendar-aware jumps (month ends, business days, etc.).

In [ ]:
ts = pd.Timestamp("2025-01-31")
ts + pd.offsets.MonthEnd(1), ts + pd.offsets.MonthBegin(1)


In [ ]:
# Business days
pd.date_range("2025-02-01", periods=10, freq="B")


## 13) Periods and converting between Period and Timestamp
Periods are handy when you care about **calendar spans** like months/quarters rather than precise instants.

In [ ]:
p = pd.Period("2025-02", freq="M")
p, p.start_time, p.end_time


In [ ]:
df["month_period"] = df["ordered_at"].dt.to_period("M")
df[["ordered_at", "month_period"]].head(10)


## 14) Interval ranges (binning continuous time)
Intervals can represent bins like hourly windows.

In [ ]:
# Create hourly bins over the data span
valid = df["ordered_at"].dropna()
bins = pd.interval_range(start=valid.min().floor("H"), end=valid.max().ceil("H"), freq="2H", closed="left")

# Assign each timestamp to a bin
df["bin_2h"] = pd.cut(df["ordered_at"], bins=bins)
df[["ordered_at", "bin_2h"]].head(12)


## 15) `merge_asof` for joining on nearest prior timestamp
Useful for event data: match each order to the latest prior price change, sensor reading, etc.

In [ ]:
# Dummy "price changes" table
price_changes = pd.DataFrame({
    "ts": pd.to_datetime(["2025-01-01 00:00", "2025-01-07 12:00", "2025-01-15 09:00", "2025-01-22 18:00"]),
    "price": [10.0, 10.5, 11.0, 10.8],
}).sort_values("ts")

orders = df[["order_id", "ordered_at", "sales"]].dropna().sort_values("ordered_at")

joined = pd.merge_asof(
    orders,
    price_changes.rename(columns={"ts":"price_ts"}).sort_values("price_ts"),
    left_on="ordered_at",
    right_on="price_ts",
    direction="backward",
)

joined.head(10)


## 16) Rounding, flooring, and ceiling timestamps
Handy for bucketing or normalization.

In [ ]:
s = df["ordered_at"].dropna().head(6)
pd.DataFrame({
    "original": s,
    "floor_15min": s.dt.floor("15min"),
    "ceil_15min": s.dt.ceil("15min"),
    "round_15min": s.dt.round("15min"),
})


## 17) Common pitfalls and best practices
- Prefer vectorized operations (`pd.to_datetime`, `.dt`, resample) over Python loops
- Use `errors='coerce'` during ingestion, then **audit** `NaT`
- Keep timestamps in **UTC** internally when integrating multiple sources; convert to local time for display
- When performance matters, specify `format=` for consistent date strings


In [ ]:
# Audit rows that failed datetime parsing
bad = df[df["ordered_at"].isna()][["order_id", "ordered_at_raw"]]
bad


## 18) Quick reference cheatsheet
```python
pd.to_datetime(s, errors='coerce', dayfirst=True)
s.dt.year / month / day / day_name() / hour / minute
df.set_index('ts').loc['2025-01':'2025-02']
df.set_index('ts').resample('D').sum()
df.set_index('ts').between_time('09:00','17:00')
s.dt.tz_localize('Europe/Dublin').dt.tz_convert('UTC')
pd.merge_asof(left, right, left_on='ts', right_on='ts', direction='backward')
s.dt.floor('15min')
```


In [ ]:
import pandas as pd
from openpyxl import load_workbook

# Example dataframe
df = pd.DataFrame({
    "timestamp": pd.to_datetime([
        "2025-02-01 14:32:10",
        "2025-02-01 18:05:22"
    ])
})

file = "output.xlsx"

# Write dataframe
df.to_excel(file, index=False)

# Open workbook to format column
wb = load_workbook(file)
ws = wb.active

# Apply datetime format to entire column A
for cell in ws["A"]:
    cell.number_format = "yyyy-mm-dd hh:mm:ss"

wb.save(file)
